# Tema: Batch frente a streaming incremental

## Objetivos
Comparar read/write con readStream/writeStream y observar un reinicio con checkpoint.

## Conceptos importantes para el examen
Batch lee un snapshot; Structured Streaming mantiene progreso. availableNow procesa lo disponible y finaliza; no es un servicio continuo. Checkpoint guarda offsets y estado, no sustituye una tabla de negocio.

**Dificultad:** Intermedio · **Tiempo estimado:** 60 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_14_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
events = spark.createDataFrame([(i, i % 4, datetime(2026, 1, 1, 10, i), float(i * 10)) for i in range(1, 13)], "event_id INT, customer_id INT, event_time TIMESTAMP, amount DOUBLE")
events.write.format("delta").mode("overwrite").saveAsTable("events_source")

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Batch

In [ ]:
spark.read.table("events_source").write.format("delta").mode("overwrite").saveAsTable("batch_copy")
print(spark.table("batch_copy").count())

### 2. Streaming finito

In [ ]:
def run_stream():
    query = (spark.readStream.table("events_source").writeStream
             .format("delta").outputMode("append")
             .option("checkpointLocation", CHECKPOINT)
             .trigger(availableNow=True).toTable("stream_copy"))
    query.awaitTermination()
    return query
query = run_stream()
assert spark.table("stream_copy").count() == 12

### 3. Reinicio con mismo checkpoint

In [ ]:
query = run_stream()
assert spark.table("stream_copy").count() == 12
print(query.lastProgress)

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Añade eventos 13–15 con INSERT a events_source; no modifiques los existentes.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Ejecuta run_stream y comprueba 15 eventos sin cambiar CHECKPOINT.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Comprueba cuántos registros tiene batch_copy antes de refrescar; luego refresca con overwrite.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Crea otro destino mediante streaming y un checkpoint distinto. Verifica que lee el snapshot inicial completo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Registra decisiones para snapshot mensual, ficheros continuos y pequeñas cargas programadas; relaciona las cuatro tecnologías.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** La fuente de streaming de este ejercicio es append-only.

**Pista 2:** Reutiliza consulta y checkpoint.

**Pista 3:** Una copia batch no se mantiene sola.

**Pista 4:** Una consulta independiente tiene checkpoint propio.

**Pista 5:** Auto Loader es una fuente de Structured Streaming; COPY INTO es una sentencia de carga.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
spark.createDataFrame([(i, i % 4, datetime(2026,1,1,10,i), float(i*10)) for i in range(13,16)], events.schema).write.format("delta").mode("append").saveAsTable("events_source")

### Solución 2

In [ ]:
query = run_stream()
assert spark.table("stream_copy").count() == 15

### Solución 3

In [ ]:
assert spark.table("batch_copy").count() == 12
spark.read.table("events_source").write.format("delta").mode("overwrite").saveAsTable("batch_copy")
assert spark.table("batch_copy").count() == 15

### Solución 4

In [ ]:
q2 = (spark.readStream.table("events_source").writeStream.format("delta").option("checkpointLocation", BASE + "/checkpoints/second").trigger(availableNow=True).toTable("stream_second"))
q2.awaitTermination()
assert spark.table("stream_second").count() == 15

### Solución 5

In [ ]:
decisions = spark.createDataFrame([
("snapshot mensual", "batch read/write"),
("archivos nuevos a gran escala", "Auto Loader + Structured Streaming"),
("pequeña carga programada de archivos", "COPY INTO"),
("tabla Delta append incremental", "Structured Streaming readStream.table")], "scenario STRING, choice STRING")
display(decisions)

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué permite continuar desde el progreso anterior?

A. Nombre del notebook

B. Checkpoint persistente

C. orderBy

D. Vista temporal

### Pregunta 2
¿Qué hace availableNow?

A. Ejecuta para siempre

B. Borra la fuente

C. Procesa los datos disponibles y termina

D. Ignora el checkpoint

### Pregunta 3
¿Qué relación existe entre Auto Loader y Structured Streaming?

A. Auto Loader puede actuar como fuente de streaming de archivos

B. Son sinónimos de COPY INTO

C. Auto Loader solo lee tablas

D. Ninguna

### Respuestas y explicación
**1. B** — Los offsets/estado guardados permiten recuperar la consulta.

**2. C** — Es útil para ingesta incremental programada.

**3. A** — El formato cloudFiles usa el motor de streaming.

## PARTE 6 - RETO FINAL
Ingiere tres lotes usando el mismo checkpoint, reinicia entre lotes y demuestra que la tabla final tiene cada evento una sola vez. Diferencia reejecución de consulta y duplicados de negocio.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
